# CSCIE89 : Timeseries Prediction using LSTM 

We will train a small LSTM neural network from scratch on timeseries data 

We use the previous **60 trading observations** of Micron Technology's (MU) opening price to predict the next opening price. Our Architecture includes four LSTM layers with 50 hidden units each, dropout of 0.2 after each layer, and a linear output with one value. 


Be sure to select the right Python virtual environment

Set `DATA_DIR` to the folder containing our CSVs.  

In [ ]:
# %pip install torch numpy pandas matplotlib scikit-learn

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DATA_DIR = Path.cwd()/"data"  # Change this if your CSV files are in another directory.
OUTPUT_DIR = Path.cwd()/"output"
LOOKBACK = 60
HIDDEN_SIZE = 50
DROPOUT = 0.2
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001
SEED = 42

# Seeding makes runs more repeatable, but different devices/backends can still
# produce different floating-point results; exact reproducibility is not promised.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# CUDA is used on supported NVIDIA GPUs, MPS on supported Macs, otherwise CPU.
# All model parameters and input tensors must be on the same device.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__}; device: {device}")
# If your installed MPS backend cannot run an operation, choose device = torch.device("cpu").

## 2. Load and clean the time series
Only `Date` and `Open` are needed. Convert non-price rows (such as dividend annotations) to missing values and remove them, then sort chronologically. Select `Open` by name rather than by its column position. We do not need to export an Excel file to train the model.

In [ ]:
def find_csv(*names):
    for name in names:
        path = DATA_DIR / name
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Expected one of {names} in {DATA_DIR.resolve()}. "
        "Copy the original CSV files there or change DATA_DIR."
    )


def load_prices(path):
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip()
    if not {"Date", "Open"}.issubset(frame.columns):
        raise ValueError(f"{path.name} must contain Date and Open columns.")
    frame = frame[["Date", "Open"]].copy()
    frame["Date"] = frame["Date"].map(lambda value: pd.to_datetime(value, errors="coerce"))
    frame["Open"] = pd.to_numeric(
        frame["Open"].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
    frame = frame.dropna(subset=["Date", "Open"])
    frame = frame.sort_values("Date").reset_index(drop=True)
    if frame.empty or not np.isfinite(frame["Open"].to_numpy()).all():
        raise ValueError(f"{path.name} contains no usable prices or contains infinite values.")
    if frame["Date"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate dates; resolve them before training.")
    return frame


mu_train = load_prices(find_csv("MU.csv"))
mu_test = load_prices(find_csv("MU_test.csv"))
if len(mu_train) <= LOOKBACK:
    raise ValueError(f"Training data needs more than {LOOKBACK} observations.")
if mu_train["Date"].max() >= mu_test["Date"].min():
    raise ValueError("All training dates must precede all test dates.")
print(f"Training rows: {len(mu_train)}; test rows: {len(mu_test)}")
print("Training dates:", mu_train["Date"].min(), "to", mu_train["Date"].max())
print("Test dates:", mu_test["Date"].min(), "to", mu_test["Date"].max())
mu_train.head()

In [ ]:
mu_train.plot(x="Date", y="Open", figsize=(10, 4), title="Micron (MU) training opening prices")
plt.ylabel("Opening price (USD)")
plt.show()

## 3. Scale prices and construct training windows
Fit the scaler **only on training prices**. Reuse this same scaler for the test data; future prices may legitimately scale outside `[0, 1]`.

For each target at position `i`, the input contains observations `i - 60` through `i - 1`. The target itself is excluded. With `N` training observations, there are `N - 60` examples.

PyTorch's LSTM with `batch_first=True` takes `(batch, time, features)`. Here each example has shape `(60, 1)`. The last dimension counts input features, not labels.

In [ ]:
mu_train_open = mu_train[["Open"]].to_numpy(dtype=np.float32)
mu_test_open = mu_test[["Open"]].to_numpy(dtype=np.float32)
scaler = MinMaxScaler(feature_range=(0, 1))
mu_train_scaled = scaler.fit_transform(mu_train_open).astype(np.float32)

# Slice all feature columns, preserving the single-feature dimension.
features_set = np.stack([
    mu_train_scaled[i - LOOKBACK:i]
    for i in range(LOOKBACK, len(mu_train_scaled))
])
labels = mu_train_scaled[LOOKBACK:].copy()
print("Inputs:", features_set.shape)  # (N - 60, 60, 1)
print("Targets:", labels.shape)       # (N - 60, 1)

## 4. Convert NumPy arrays into PyTorch batches
`TensorDataset` pairs each window with its target. `DataLoader` yields mini-batches. Shuffling complete training windows is valid here: the order **inside** each window is preserved and hidden states are reset for every forward pass. We never randomly mix training and test dates.

In [ ]:
# float32 matches the default dtype of the model parameters. NumPy often uses
# float64, so explicitly converting prevents a common dtype mismatch.
X_train = torch.from_numpy(features_set).float()
y_train = torch.from_numpy(labels).float()
train_dataset = TensorDataset(X_train, y_train)

# A dedicated random generator makes the order of shuffled batches repeatable.
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    num_workers=0,  # Works directly in notebook kernels without worker setup.
)
# Tensors stay on CPU until each batch is moved to the chosen device.
print(f"{len(train_dataset)} examples in {len(train_loader)} batches per epoch")

## 5. Define the four-layer LSTM

`nn.LSTM` returns `(sequence_output, (h_n, c_n))`. `sequence_output` contains hidden outputs for all time steps; `h_n` and `c_n` are the final hidden and cell states. No initial states are passed, so PyTorch initializes them to zero on every call. This is a unidirectional, stateless-across-batches model. The architecture is equivalent to the original, but initialization and backend details differ, so training results will not be numerically identical.

In [ ]:
class MuLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, dropout=0.2):
        super().__init__()  # Registers child layers and their trainable parameters.
        # input_size counts features at each time step, not sequence length.
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.lstm2 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.lstm3 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.lstm4 = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.dropout4 = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (B, 60, 1), where B may be smaller in the final mini-batch.
        # Each LSTM produces (B, 60, 50). '_' discards its final state tuple.
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x)
        x, _ = self.lstm3(x)
        x = self.dropout3(x)
        x, _ = self.lstm4(x)

        # The last time step summarizes the window: (B, 60, 50) -> (B, 50).
        # Select the last TIME step, not the last example in the batch.
        x = x[:, -1, :]
        x = self.dropout4(x)
        # Linear regression head: (B, 50) -> (B, 1). No sigmoid is applied,
        # because future scaled prices can lie outside the training range.
        return self.output(x)


model_config = {"input_size": 1, "hidden_size": HIDDEN_SIZE, "dropout": DROPOUT}
model = MuLSTM(**model_config).to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Train with an explicit PyTorch loop
The loss is mean squared error on **scaled prices**. Adam updates the model parameters. Unlike Keras `fit`, PyTorch exposes each step: clear old gradients, predict, compute loss, backpropagate, and update weights.

Targets and predictions must both have shape `(B, 1)`. A target of shape `(B,)` can accidentally broadcast into a `(B, B)` error matrix. The assertion below guards against this.

We retain the original 100 epochs and batch size of 32. There is no validation set in this conversion; training loss alone cannot establish generalization. If tuning hyperparameters, hold out the latest part of the training period for validation and fit the scaler on the earlier portion only, leaving the test set untouched.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {"loss": []}  

for epoch in range(EPOCHS):
    model.train()  # Enables dropout; this does not itself perform training.
    total_squared_error = 0.0
    examples_seen = 0

    for batch_x, batch_y in train_loader:
        # Inputs, targets, and model parameters must share a device.
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Gradients accumulate by default in PyTorch. Clear the previous batch's
        # gradients before calculating new ones; None can avoid unnecessary fills.
        optimizer.zero_grad(set_to_none=True)
        batch_predictions = model(batch_x)  # Calls forward and builds an autograd graph.
        assert batch_predictions.shape == batch_y.shape
        loss = criterion(batch_predictions, batch_y)
        loss.backward()   # Compute gradients d(loss)/d(parameter) using autograd.
        optimizer.step()  # Apply Adam's parameter updates using those gradients.

        # item() detaches this scalar from autograd. Weight batch mean losses by
        # batch size so a small final batch does not count as a full-size batch.
        batch_count = batch_x.size(0)
        total_squared_error += loss.item() * batch_count
        examples_seen += batch_count

    epoch_loss = total_squared_error / examples_seen
    history["loss"].append(epoch_loss)
    if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
        print(f"Epoch {epoch + 1:3d}/{EPOCHS}: training MSE = {epoch_loss:.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history["loss"]) + 1), history["loss"], label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE (scaled prices)")
plt.title("Training loss")
plt.legend()
plt.show()

## 7. Prepare chronological test windows
This is **rolling one-step-ahead evaluation**. The first test prediction uses the last 60 training observations. Each later prediction uses actual opening prices observed before that test day, including earlier test observations. It is not a recursive forecast of the entire test period made on the last training day.

Use the training scaler without refitting it. Window lengths are derived from the CSV contents, replacing the original hard-coded 1,975 training rows and 37 test rows.

In [ ]:
context_prices = np.concatenate([mu_train_open[-LOOKBACK:], mu_test_open], axis=0)
context_scaled = scaler.transform(context_prices).astype(np.float32)
test_features = np.stack([
    context_scaled[i:i + LOOKBACK]
    for i in range(len(mu_test_open))
])
# Window i stops immediately before target i; it never includes that day's price.
X_test = torch.from_numpy(test_features).float()
test_loader = DataLoader(TensorDataset(X_test), batch_size=BATCH_SIZE, shuffle=False)
print("Test inputs:", X_test.shape)  # (number of test observations, 60, 1)

## 8. Predict and convert back to dollars
`model.eval()` disables dropout. It does **not** disable gradient tracking; `torch.inference_mode()` handles that separately. Move predictions back to CPU before converting to NumPy for scikit-learn and plotting.

In [ ]:
model.eval()
prediction_batches = []
with torch.inference_mode():
    for (batch_x,) in test_loader:
        batch_predictions = model(batch_x.to(device))
        prediction_batches.append(batch_predictions.cpu().numpy())

scaled_predictions = np.concatenate(prediction_batches, axis=0)
predictions = scaler.inverse_transform(scaled_predictions)
assert predictions.shape == mu_test_open.shape

# A persistence baseline predicts that the next opening price equals the last
# observed opening price. Compare against it rather than judging only a plot.
baseline = context_prices[LOOKBACK - 1:-1]
for name, values in [("LSTM", predictions), ("Previous-open baseline", baseline)]:
    errors = values - mu_test_open
    print(f"{name}: MAE = ${np.abs(errors).mean():.2f}; RMSE = ${np.sqrt(np.mean(errors ** 2)):.2f}")

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(mu_test["Date"], mu_test_open[:, 0], color="blue", label="Actual opening price")
plt.plot(mu_test["Date"], predictions[:, 0], color="red", label="LSTM prediction")
plt.plot(mu_test["Date"], baseline[:, 0], color="gray", linestyle="--", label="Previous-open baseline")
plt.title("Micron (MU) opening prices: rolling one-step-ahead predictions")
plt.xlabel("Date")
plt.ylabel("Opening price (USD)")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. Save and reload the PyTorch model
A `state_dict` stores learned parameter tensors. Reconstruct the same architecture before loading it. The checkpoint also stores the lookback, model configuration, training history, and fitted scaler parameters, which are needed to interpret predictions consistently.

The checkpoint below supports inference; resuming Adam training exactly would additionally require optimizer and random-generator states. Load only checkpoints you trust.

In [ ]:
checkpoint_path = OUTPUT_DIR / "mu_lstm_pytorch.pt"
checkpoint = {
    # Save CPU copies for portability across CPU, CUDA, and MPS environments.
    "model_state_dict": {key: value.detach().cpu().clone() for key, value in model.state_dict().items()},
    "model_config": model_config,
    "lookback": LOOKBACK,
    "history": history,
    # Lists and primitive values can be loaded with weights_only=True.
    "scaler": {
        "feature_range": list(scaler.feature_range),
        "scale_": scaler.scale_.tolist(),
        "min_": scaler.min_.tolist(),
        "data_min_": scaler.data_min_.tolist(),
        "data_max_": scaler.data_max_.tolist(),
        "data_range_": scaler.data_range_.tolist(),
        "n_features_in_": int(scaler.n_features_in_),
        "n_samples_seen_": int(scaler.n_samples_seen_),
    },
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, checkpoint_path)
print("Saved:", checkpoint_path.resolve())

In [ ]:
# Load tensors on CPU first, then move the reconstructed model to your device.
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
reloaded_model = MuLSTM(**saved["model_config"])
reloaded_model.load_state_dict(saved["model_state_dict"])
reloaded_model = reloaded_model.to(device)
reloaded_model.eval()  # New modules start in training mode, so disable dropout again.

reloaded_scaler = MinMaxScaler(feature_range=tuple(saved["scaler"]["feature_range"]))
for name in ("scale_", "min_", "data_min_", "data_max_", "data_range_"):
    setattr(reloaded_scaler, name, np.asarray(saved["scaler"][name], dtype=np.float32))
reloaded_scaler.n_features_in_ = saved["scaler"]["n_features_in_"]
reloaded_scaler.n_samples_seen_ = saved["scaler"]["n_samples_seen_"]

# Verify that saving/loading preserved predictions for the same input batch.
with torch.inference_mode():
    sample = X_test[:BATCH_SIZE].to(device)
    expected = model(sample)
    actual = reloaded_model(sample)
    torch.testing.assert_close(actual, expected)
np.testing.assert_allclose(
    reloaded_scaler.inverse_transform(scaled_predictions), predictions
)
print("Reloaded model and scaler reproduce the original predictions.")

## Interpretation
A curve that follows the trend does not by itself establish useful forecasting skill. Compare test errors with the persistence baseline. This example uses only past opening prices and does not assess trading returns, transaction costs, or information from other features. Predictions for later test days use the actual earlier test prices, as explained above.